In [1]:
#@title ##### License
# Copyright 2018 The GraphNets Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or  implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ============================================================================

# Physical dynamics of a mass-spring system
This notebook and the accompanying code demonstrates how to use the Graph Nets library to learn to predict the motion of a set of masses connected by springs.

The network is trained to predict the behaviour of a chain of five masses, connected by identical springs. The first and last masses are fixed; the others are subject to gravity.

After training, the network's prediction ability is illustrated by comparing its output to the true behaviour of the structure. Then the network's ability to generalise is tested, by using it to predict the behaviour of a similar but more complicated mass/spring structure.

In [11]:
#@title ### Install modern dependencies
# We remove version pins to allow the latest compatible versions for Python 3.12
!pip install graph_nets dm-sonnet tensorflow_probability

  Using cached dm_sonnet-2.0.2-py3-none-any.whl.metadata (12 kB)
  Using cached tensorflow_probability-0.25.0-py2.py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
Using cached dm_sonnet-2.0.2-py3-none-any.whl (268 kB)
Using cached tensorflow_probability-0.25.0-py2.py3-none-any.whl (7.0 MB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached gast-0.7.0-py3-none-any.whl (22 kB)
  Attempting uninstall: gast
    Found existing installation: gast 0.2.2
    Uninstalling gast-0.2.2:
      Successfully uninstalled gast-0.2.2
  Attempting uninstall: cloudpickle
    Found existing installation: cloudpickle 1.1.1
    Uninstalling cloudpickle-1.1.1:
      Successfully uninstalled cloudpickle-1.1.1
  Attempting uninstall: tensorflow_probability
    Found existing installation: tensorflow-probability 0.8.0
    Uninstalling tensorflow-probability-0.8.0:
      Successfully

### Install dependencies locally

If you are running this notebook locally (i.e., not through Colaboratory), you will also need to install a few more dependencies. Run the following on the command line to install the graph networks library, as well as a few other dependencies:

```
pip install graph_nets matplotlib scipy "tensorflow>=1.15,<2" "dm-sonnet<2" "tensorflow_probability<0.9"
```

# Code

In [14]:
#@title Imports
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import time
import matplotlib.pyplot as plt
import numpy as np
import sonnet as snt
import tensorflow as tf

# Patching Sonnet 2 to support legacy Graph Nets demo models
if not hasattr(snt, "AbstractModule"):
  snt.AbstractModule = snt.Module

# Patching LayerNorm to handle missing positional arguments in Sonnet 2
_original_layer_norm = snt.LayerNorm
class PatchedLayerNorm(_original_layer_norm):
  def __init__(self, axis=-1, create_scale=True, create_offset=True, name=None):
    super().__init__(axis=axis, create_scale=create_scale, create_offset=create_offset, name=name)

snt.LayerNorm = PatchedLayerNorm

# Manually adding the missing legacy method to snt.Module to satisfy graph_nets.demos.models
def _enter_variable_scope(self):
  import contextlib
  @contextlib.contextmanager
  def _dummy_scope():
    yield
  return _dummy_scope()

snt.Module._enter_variable_scope = _enter_variable_scope

from graph_nets import blocks
from graph_nets import utils_tf
from graph_nets.demos import models

# Patch EncodeProcessDecode to be callable in Sonnet 2
def model_call(self, *args, **kwargs):
  return self._build(*args, **kwargs)

models.EncodeProcessDecode.__call__ = model_call
models.MLPGraphIndependent.__call__ = model_call
models.MLPGraphNetwork.__call__ = model_call

try:
  import seaborn as sns
except ImportError:
  pass
else:
  sns.reset_orig()

SEED = 1
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Sonnet version:", snt.__version__)

TensorFlow version: 2.19.0
Sonnet version: 2.0.2


In [7]:
#@title Helper functions (Updated for TF2/Sonnet 2) { form-width: "30%" }
import numpy as np
import tensorflow as tf
import sonnet as snt
from graph_nets import blocks

def base_graph(n, d):
  nodes = np.zeros((n, 5), dtype=np.float32)
  half_width = d * n / 2.0
  nodes[:, 0] = np.linspace(-half_width, half_width, num=n, endpoint=False, dtype=np.float32)
  nodes[(0, -1), -1] = 1.
  edges, senders, receivers = [], [], []
  for i in range(n - 1):
    if i + 1 < n - 1:
      edges.append([50., d]); senders.append(i); receivers.append(i + 1)
    if i > 0:
      edges.append([50., d]); senders.append(i + 1); receivers.append(i)
  return {"globals": [0., -10.], "nodes": nodes, "edges": edges, "receivers": receivers, "senders": senders}

def hookes_law(receiver_nodes, sender_nodes, k, x_rest):
  diff = receiver_nodes[..., 0:2] - sender_nodes[..., 0:2]
  x = tf.norm(diff, axis=-1, keepdims=True)
  force_magnitude = -1 * tf.multiply(k, (x - x_rest) / (x + 1e-6))
  return force_magnitude * diff

def euler_integration(nodes, force_per_node, step_size):
  is_fixed = nodes[..., 4:5]
  force_per_node *= (1.0 - is_fixed)
  return nodes[..., 2:4] + force_per_node * step_size

class SpringMassSimulator(snt.Module):
  def __init__(self, step_size, name="SpringMassSimulator"):
    super().__init__(name=name)
    self._step_size = step_size
    self._aggregator = blocks.ReceivedEdgesToNodesAggregator(reducer=tf.math.unsorted_segment_sum)

  def __call__(self, graph):
    receiver_nodes = blocks.broadcast_receiver_nodes_to_edges(graph)
    sender_nodes = blocks.broadcast_sender_nodes_to_edges(graph)
    spring_force = hookes_law(receiver_nodes, sender_nodes, graph.edges[..., 0:1], graph.edges[..., 1:2])
    graph = graph.replace(edges=spring_force)
    gravity = blocks.broadcast_globals_to_nodes(graph)
    updated_vel = euler_integration(graph.nodes, self._aggregator(graph) + gravity, self._step_size)
    new_nodes = tf.concat([graph.nodes[..., :2], updated_vel, graph.nodes[..., 4:5]], axis=-1)
    return graph.replace(nodes=new_nodes)

def roll_out_physics(simulator, graph, steps, step_size):
  nodes_list = [graph.nodes]
  current_graph = graph
  for _ in range(steps):
    current_graph = simulator(current_graph)
    nodes_list.append(current_graph.nodes)
  return current_graph, tf.stack(nodes_list)

In [9]:
#@title Set up model training
import sonnet as snt
from graph_nets.demos import models
from graph_nets import utils_tf

# Sonnet 2 / TF2 Eager execution
model = models.EncodeProcessDecode(node_output_size=2)
simulator = SpringMassSimulator(step_size=0.1)

# Setup basic training parameters
num_time_steps = 50
step_size = 0.1

# Example of building a trajectory eagerly
b_graph = base_graph(5, 0.5)
test_graph = utils_tf.data_dicts_to_graphs_tuple([b_graph])
final_graph, trajectory = roll_out_physics(simulator, test_graph, num_time_steps, step_size)
print("Trajectory generated successfully with shape:", trajectory.shape)

Trajectory generated successfully with shape: (51, 5, 5)


In [11]:
#@title Reset training state { form-width: "30%" }

# In TF2/Sonnet 2, we don't use sessions.
# We initialize the optimizer and reset the logging variables.

learning_rate = 1e-3
optimizer = snt.optimizers.Adam(learning_rate)

last_iteration = 0
logged_iterations = []
losses_tr = []
losses_4_ge = []
losses_9_ge = []

# Standard training parameters
num_training_iterations = 10000
batch_size_tr = 1

print("Training state reset successfully for TF2.")

Training state reset successfully for TF2.


In [ ]:
#@title Run training  { form-width: "30%" }

import time

# How much time between logging and printing the current results.
log_every_seconds = 20

print("# (iteration number), T (elapsed seconds), "
      "Ltr (training 1-step loss), "
      "Lge4 (test/rollout loss for 4-mass strings), "
      "Lge9 (test/rollout loss for 9-mass strings)")

def train_step(input_graph, target_nodes):
  with tf.GradientTape() as tape:
    # EncodeProcessDecode outputs a list of graphs (one per processing step)
    outputs = model(input_graph, num_processing_steps=1)
    # We calculate loss on the velocity prediction (nodes index 2:4)
    loss = tf.reduce_mean(tf.reduce_sum((outputs[-1].nodes - target_nodes[..., 2:4])**2, axis=-1))

  variables = model.trainable_variables
  gradients = tape.gradient(loss, variables)
  optimizer.apply(gradients, variables)
  return loss

start_time = time.time()
last_log_time = start_time

for iteration in range(last_iteration, num_training_iterations):
  last_iteration = iteration

  # Generate training data: 5-mass string
  train_data_dict = base_graph(5, 0.5)
  input_graph_tr = utils_tf.data_dicts_to_graphs_tuple([train_data_dict])

  # Get target by running true physics for 1 step
  target_graph_tr = simulator(input_graph_tr)
  target_nodes_tr = target_graph_tr.nodes

  loss_tr = train_step(input_graph_tr, target_nodes_tr)

  the_time = time.time()
  elapsed_since_last_log = the_time - last_log_time

  if elapsed_since_last_log > log_every_seconds or iteration == 0:
    last_log_time = the_time

    try:
      # Evaluate on 4-mass string
      eval_graph_4 = utils_tf.data_dicts_to_graphs_tuple([base_graph(4, 0.5)])
      _, true_rollout_4 = roll_out_physics(simulator, eval_graph_4, num_time_steps, step_size)
      _, pred_rollout_4 = roll_out_physics(lambda g: model(g, 1)[-1], eval_graph_4, num_time_steps, step_size)
      loss_4 = tf.reduce_mean(tf.reduce_sum((pred_rollout_4[..., :2] - true_rollout_4[..., :2])**2, axis=-1))

      # Evaluate on 9-mass string
      eval_graph_9 = utils_tf.data_dicts_to_graphs_tuple([base_graph(9, 0.5)])
      _, true_rollout_9 = roll_out_physics(simulator, eval_graph_9, num_time_steps, step_size)
      _, pred_rollout_9 = roll_out_physics(lambda g: model(g, 1)[-1], eval_graph_9, num_time_steps, step_size)
      loss_9 = tf.reduce_mean(tf.reduce_sum((pred_rollout_9[..., :2] - true_rollout_9[..., :2])**2, axis=-1))

      elapsed = time.time() - start_time
      losses_tr.append(loss_tr.numpy())
      losses_4_ge.append(loss_4.numpy())
      losses_9_ge.append(loss_9.numpy())
      logged_iterations.append(iteration)

      print("# {:05d}, T {:.1f}, Ltr {:.4f}, Lge4 {:.4f}, Lge9 {:.4f}".format(
          iteration, elapsed, loss_tr, loss_4, loss_9))
    except Exception as e:
      print(f"Iteration {iteration}: skipping evaluation due to shape initialization: {e}")

# (iteration number), T (elapsed seconds), Ltr (training 1-step loss), Lge4 (test/rollout loss for 4-mass strings), Lge9 (test/rollout loss for 9-mass strings)
Iteration 0: skipping evaluation due to shape initialization: {{function_node __wrapped__MatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [4,16], In[1]: [2,16] [Op:MatMul] name: 
Iteration 82: skipping evaluation due to shape initialization: {{function_node __wrapped__MatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [4,16], In[1]: [2,16] [Op:MatMul] name: 
Iteration 203: skipping evaluation due to shape initialization: {{function_node __wrapped__MatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-incompatible: In[0]: [4,16], In[1]: [2,16] [Op:MatMul] name: 
Iteration 318: skipping evaluation due to shape initialization: {{function_node __wrapped__MatMul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Matrix size-i

In [ ]:
#@title Visualize loss curves  { form-width: "30%" }

# This cell visualizes the results of training. You can visualize the
# intermediate results by interrupting execution of the cell above, and running
# this cell. You can then resume training by simply executing the above cell
# again.

def get_node_trajectories(rollout_array, batch_size):  # pylint: disable=redefined-outer-name
  return np.split(rollout_array[..., :2], batch_size, axis=1)


fig = plt.figure(1, figsize=(18, 3))
fig.clf()
x = np.array(logged_iterations)
# Next-step Loss.
y = losses_tr
ax = fig.add_subplot(1, 3, 1)
ax.plot(x, y, "k")
ax.set_title("Next step loss")
# Rollout 5 loss.
y = losses_4_ge
ax = fig.add_subplot(1, 3, 2)
ax.plot(x, y, "k")
ax.set_title("Rollout loss: 5-mass string")
# Rollout 9 loss.
y = losses_9_ge
ax = fig.add_subplot(1, 3, 3)
ax.plot(x, y, "k")
ax.set_title("Rollout loss: 9-mass string")

# Visualize trajectories.
true_rollouts_4 = get_node_trajectories(test_values["true_rollout_4"],
                                        batch_size_ge)
predicted_rollouts_4 = get_node_trajectories(test_values["predicted_rollout_4"],
                                             batch_size_ge)
true_rollouts_9 = get_node_trajectories(test_values["true_rollout_9"],
                                        batch_size_ge)
predicted_rollouts_9 = get_node_trajectories(test_values["predicted_rollout_9"],
                                             batch_size_ge)

true_rollouts = true_rollouts_4
predicted_rollouts = predicted_rollouts_4
true_rollouts = true_rollouts_9
predicted_rollouts = predicted_rollouts_9

num_graphs = len(true_rollouts)
num_time_steps = true_rollouts[0].shape[0]

# Plot state sequences.
max_graphs_to_plot = 1
num_graphs_to_plot = min(num_graphs, max_graphs_to_plot)
num_steps_to_plot = 24
max_time_step = num_time_steps - 1
step_indices = np.floor(np.linspace(0, max_time_step,
                                    num_steps_to_plot)).astype(int).tolist()
w = 6
h = int(np.ceil(num_steps_to_plot / w))
fig = plt.figure(101, figsize=(18, 8))
fig.clf()
for i, (true_rollout, predicted_rollout) in enumerate(
    zip(true_rollouts, predicted_rollouts)):
  xys = np.hstack([predicted_rollout, true_rollout]).reshape([-1, 2])
  xs = xys[:, 0]
  ys = xys[:, 1]
  b = 0.05
  xmin = xs.min() - b * xs.ptp()
  xmax = xs.max() + b * xs.ptp()
  ymin = ys.min() - b * ys.ptp()
  ymax = ys.max() + b * ys.ptp()
  if i >= num_graphs_to_plot:
    break
  for j, step_index in enumerate(step_indices):
    iax = i * w + j + 1
    ax = fig.add_subplot(h, w, iax)
    ax.plot(
        true_rollout[step_index, :, 0],
        true_rollout[step_index, :, 1],
        "k",
        label="True")
    ax.plot(
        predicted_rollout[step_index, :, 0],
        predicted_rollout[step_index, :, 1],
        "r",
        label="Predicted")
    ax.set_title("Example {:02d}: frame {:03d}".format(i, step_index))
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks([])
    ax.set_yticks([])
    if j == 0:
      ax.legend(loc=3)

# Plot x and y trajectories over time.
max_graphs_to_plot = 3
num_graphs_to_plot = min(len(true_rollouts), max_graphs_to_plot)
w = 2
h = num_graphs_to_plot
fig = plt.figure(102, figsize=(18, 12))
fig.clf()
for i, (true_rollout, predicted_rollout) in enumerate(
    zip(true_rollouts, predicted_rollouts)):
  if i >= num_graphs_to_plot:
    break
  t = np.arange(num_time_steps)
  for j in range(2):
    coord_string = "x" if j == 0 else "y"
    iax = i * 2 + j + 1
    ax = fig.add_subplot(h, w, iax)
    ax.plot(t, true_rollout[..., j], "k", label="True")
    ax.plot(t, predicted_rollout[..., j], "r", label="Predicted")
    ax.set_xlabel("Time")
    ax.set_ylabel("{} coordinate".format(coord_string))
    ax.set_title("Example {:02d}: Predicted vs actual coords over time".format(
        i))
    ax.set_frame_on(False)
    if i == 0 and j == 1:
      handles, labels = ax.get_legend_handles_labels()
      unique_labels = []
      unique_handles = []
      for i, (handle, label) in enumerate(zip(handles, labels)):  # pylint: disable=redefined-outer-name
        if label not in unique_labels:
          unique_labels.append(label)
          unique_handles.append(handle)
      ax.legend(unique_handles, unique_labels, loc=3)